# Task 4: Optimize Portfolio Based on Forecast

## Objective
Use insights from the forecast to construct an optimal portfolio using Modern Portfolio Theory (MPT). Combine Tesla forecast with historical data for other assets to build an optimized portfolio.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

# Portfolio optimization
from pypfopt import EfficientFrontier, risk_models, expected_returns
from pypfopt.discrete_allocation import DiscreteAllocation, get_latest_prices
from scipy.optimize import minimize

plt.style.use('seaborn-v0_8-darkgrid')

## 1. Load Data and Forecasts

In [ ]:
# Load processed data for all assets
tsla_data = pd.read_csv('../data/processed/tsla_processed.csv', index_col=0, parse_dates=True)
bnd_data = pd.read_csv('../data/processed/bnd_processed.csv', index_col=0, parse_dates=True)
spy_data = pd.read_csv('../data/processed/spy_processed.csv', index_col=0, parse_dates=True)

# Load forecast summary
with open('../data/processed/forecast_summary.json', 'r') as f:
    forecast_summary = json.load(f)

print("Data loaded successfully!")
print(f"TSLA shape: {tsla_data.shape}")
print(f"BND shape: {bnd_data.shape}")
print(f"SPY shape: {spy_data.shape}")

## 2. Prepare Expected Returns

In [ ]:
# Calculate daily returns for all assets
tsla_returns = tsla_data['Close'].pct_change().dropna()
bnd_returns = bnd_data['Close'].pct_change().dropna()
spy_returns = spy_data['Close'].pct_change().dropna()

# Align dates
common_dates = tsla_returns.index.intersection(bnd_returns.index).intersection(spy_returns.index)
tsla_returns = tsla_returns.loc[common_dates]
bnd_returns = bnd_returns.loc[common_dates]
spy_returns = spy_returns.loc[common_dates]

# Create returns dataframe
returns_df = pd.DataFrame({
    'TSLA': tsla_returns,
    'BND': bnd_returns,
    'SPY': spy_returns
})

print(f"Returns dataframe shape: {returns_df.shape}")
print(f"Date range: {returns_df.index.min()} to {returns_df.index.max()}")
print(f"\nFirst few rows:")
print(returns_df.head())

In [ ]:
# TSLA: Use forecasted return (annualized from 12-month forecast)
current_tsla_price = tsla_data['Close'].iloc[-1]
forecast_tsla_price = forecast_summary['Forecast_12M']
tsla_forecast_return = ((forecast_tsla_price / current_tsla_price) - 1)  # Annual return
tsla_expected_daily_return = tsla_forecast_return / 252  # Convert to daily

# BND and SPY: Use historical average daily returns (annualized)
bnd_annual_return = bnd_returns.mean() * 252
spy_annual_return = spy_returns.mean() * 252
bnd_expected_daily_return = bnd_returns.mean()
spy_expected_daily_return = spy_returns.mean()

# Create expected returns vector (daily)
expected_returns_daily = pd.Series({
    'TSLA': tsla_expected_daily_return,
    'BND': bnd_expected_daily_return,
    'SPY': spy_expected_daily_return
})

# Annualized expected returns
expected_returns_annual = expected_returns_daily * 252

print("="*60)
print("EXPECTED RETURNS")
print("="*60)
print(f"\nDaily Expected Returns:")
print(expected_returns_daily)
print(f"\nAnnualized Expected Returns:")
print(expected_returns_annual)
print(f"\nTSLA Forecast Details:")
print(f"  Current Price: ${current_tsla_price:.2f}")
print(f"  Forecasted Price (12M): ${forecast_tsla_price:.2f}")
print(f"  Expected Annual Return: {tsla_forecast_return*100:.2f}%")

## 3. Compute Covariance Matrix

In [ ]:
# Calculate covariance matrix (annualized)
cov_matrix = returns_df.cov() * 252

print("="*60)
print("COVARIANCE MATRIX (Annualized)")
print("="*60)
print(cov_matrix)

# Visualize covariance matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cov_matrix, annot=True, fmt='.4f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Covariance Matrix (Annualized)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/covariance_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Generate Efficient Frontier

In [ ]:
# Create Efficient Frontier
ef = EfficientFrontier(expected_returns_annual, cov_matrix)

# Generate efficient frontier
ef_min_vol = EfficientFrontier(expected_returns_annual, cov_matrix)
ef_max_sharpe = EfficientFrontier(expected_returns_annual, cov_matrix)

# Find minimum volatility portfolio
min_vol_weights = ef_min_vol.min_volatility()
min_vol_perf = ef_min_vol.portfolio_performance(verbose=False)

# Find maximum Sharpe ratio portfolio (tangency portfolio)
max_sharpe_weights = ef_max_sharpe.max_sharpe()
max_sharpe_perf = ef_max_sharpe.portfolio_performance(verbose=False)

print("="*60)
print("OPTIMAL PORTFOLIOS")
print("="*60)

print(f"\nMinimum Volatility Portfolio:")
print(f"  Weights: {min_vol_weights}")
print(f"  Expected Annual Return: {min_vol_perf[0]*100:.2f}%")
print(f"  Annual Volatility: {min_vol_perf[1]*100:.2f}%")
print(f"  Sharpe Ratio: {min_vol_perf[2]:.4f}")

print(f"\nMaximum Sharpe Ratio Portfolio (Tangency Portfolio):")
print(f"  Weights: {max_sharpe_weights}")
print(f"  Expected Annual Return: {max_sharpe_perf[0]*100:.2f}%")
print(f"  Annual Volatility: {max_sharpe_perf[1]*100:.2f}%")
print(f"  Sharpe Ratio: {max_sharpe_perf[2]:.4f}")

In [ ]:
# Generate efficient frontier points
ef = EfficientFrontier(expected_returns_annual, cov_matrix)
ef.efficient_return(0.15)  # Set target return to generate frontier

# Get efficient frontier
ef_dict = ef.efficient_frontier(100)
ef_returns = [x[0] for x in ef_dict.values()]
ef_volatilities = [x[1] for x in ef_dict.values()]

# Plot efficient frontier
fig, ax = plt.subplots(figsize=(12, 8))

# Plot efficient frontier
ax.plot(ef_volatilities, ef_returns, 'b-', linewidth=2, label='Efficient Frontier')

# Plot individual assets
asset_vols = np.sqrt(np.diag(cov_matrix))
ax.scatter(asset_vols, expected_returns_annual.values, 
          s=200, alpha=0.7, c=['red', 'blue', 'green'], 
          label=['TSLA', 'BND', 'SPY'])
for i, ticker in enumerate(['TSLA', 'BND', 'SPY']):
    ax.annotate(ticker, (asset_vols[i], expected_returns_annual.values[i]),
                fontsize=12, fontweight='bold')

# Plot minimum volatility portfolio
min_vol_vol = min_vol_perf[1]
min_vol_ret = min_vol_perf[0]
ax.scatter(min_vol_vol, min_vol_ret, s=300, marker='*', 
          color='gold', edgecolors='black', linewidth=2,
          label='Minimum Volatility Portfolio', zorder=5)

# Plot maximum Sharpe ratio portfolio
max_sharpe_vol = max_sharpe_perf[1]
max_sharpe_ret = max_sharpe_perf[0]
ax.scatter(max_sharpe_vol, max_sharpe_ret, s=300, marker='*', 
          color='purple', edgecolors='black', linewidth=2,
          label='Maximum Sharpe Ratio Portfolio', zorder=5)

ax.set_xlabel('Annual Volatility (Risk)', fontsize=14)
ax.set_ylabel('Expected Annual Return', fontsize=14)
ax.set_title('Efficient Frontier', fontsize=16, fontweight='bold')
ax.legend(fontsize=11, loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/processed/efficient_frontier.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Portfolio Recommendation

In [ ]:
# Select optimal portfolio (Maximum Sharpe Ratio for best risk-adjusted returns)
optimal_weights = max_sharpe_weights
optimal_performance = max_sharpe_perf

print("="*60)
print("RECOMMENDED OPTIMAL PORTFOLIO")
print("="*60)
print(f"\nPortfolio Weights:")
for asset, weight in optimal_weights.items():
    print(f"  {asset}: {weight*100:.2f}%")

print(f"\nPortfolio Performance:")
print(f"  Expected Annual Return: {optimal_performance[0]*100:.2f}%")
print(f"  Annual Volatility (Risk): {optimal_performance[1]*100:.2f}%")
print(f"  Sharpe Ratio: {optimal_performance[2]:.4f}")

# Create portfolio summary
portfolio_summary = {
    'TSLA_Weight': float(optimal_weights['TSLA']),
    'BND_Weight': float(optimal_weights['BND']),
    'SPY_Weight': float(optimal_weights['SPY']),
    'Expected_Annual_Return': float(optimal_performance[0]),
    'Annual_Volatility': float(optimal_performance[1]),
    'Sharpe_Ratio': float(optimal_performance[2])
}

# Save portfolio recommendation
with open('../data/processed/optimal_portfolio.json', 'w') as f:
    json.dump(portfolio_summary, f, indent=4)

print(f"\nPortfolio recommendation saved to: ../data/processed/optimal_portfolio.json")

In [ ]:
# Visualize portfolio weights
fig, ax = plt.subplots(figsize=(10, 8))

weights_df = pd.Series(optimal_weights)
colors = ['#e74c3c', '#3498db', '#2ecc71']
bars = ax.bar(weights_df.index, weights_df.values * 100, color=colors, alpha=0.7, edgecolor='black', linewidth=2)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}%',
            ha='center', va='bottom', fontsize=14, fontweight='bold')

ax.set_ylabel('Portfolio Weight (%)', fontsize=14)
ax.set_title('Optimal Portfolio Allocation', fontsize=16, fontweight='bold')
ax.set_ylim(0, max(weights_df.values) * 100 * 1.2)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../data/processed/portfolio_weights.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Portfolio Justification

### Rationale for Portfolio Selection

The **Maximum Sharpe Ratio Portfolio** (Tangency Portfolio) has been selected as the optimal portfolio because:

1. **Risk-Adjusted Returns**: The Sharpe Ratio measures excess return per unit of risk. A higher Sharpe Ratio indicates better risk-adjusted performance, which is crucial for portfolio optimization.

2. **Balanced Approach**: This portfolio balances the high-growth potential of TSLA (based on our forecast) with the stability of BND and diversification of SPY.

3. **Forecast Integration**: The portfolio incorporates our TSLA price forecast, which provides a forward-looking view rather than relying solely on historical data.

4. **Efficient Frontier**: This portfolio lies on the efficient frontier, meaning it offers the highest expected return for its level of risk, or equivalently, the lowest risk for its expected return.

### Portfolio Characteristics

- **TSLA Allocation**: [X]% - Captures forecasted growth potential
- **BND Allocation**: [X]% - Provides stability and income
- **SPY Allocation**: [X]% - Offers broad market diversification

This allocation strategy aims to optimize returns while managing risk through diversification across different asset classes with varying risk profiles.